RAG Pipeline - Data Ingestion to vector DB Pipeline

In [41]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [42]:
# Read all the pdf's inside the directory

def process_pdfs(pdf_dict):
    all_doc = []
    pdf_dir = Path(pdf_dict)

    #find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF filess to process")

    for pf in pdf_files:
        print(f"\nProcessing: {pf.name}")
        try: 
            loader = PyPDFLoader(str(pf))
            doc = loader.load()

            for d in doc:
                d.metadata['source_file']=pf.name
                d.metadata['file_type'] = 'pdf'

            all_doc.extend(doc)
            print(f"✔️ Loaded {len(doc)} pages")
        
        except Exception as e:
            print(f"✖️ Error:{e}")

    print(f"\n Total documents loaded: {len(all_doc)}")
    return all_doc

#Process all PDFs in the data directory
all_pdf_doc = process_pdfs("../data")



Found 3 PDF filess to process

Processing: ABHI.pdf
✔️ Loaded 33 pages

Processing: Sutherland Job Description.pdf
✔️ Loaded 1 pages

Processing: Major Document.pdf
✔️ Loaded 78 pages

 Total documents loaded: 112


In [43]:
all_pdf_doc

[Document(metadata={'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-11T14:47:25+05:30', 'moddate': '2025-12-12T16:24:36-05:00', 'ieee article id': '11269767', 'trapped': 'False', 'ieee issue id': '10820123', 'subject': 'IEEE Access;2025;13; ;10.1109/ACCESS.2025.3637567', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publication id': '6287639', 'title': 'Rescue Robots for Casualty Extraction: A Comprehensive Review', 'source': '../data/pdf/ABHI.pdf', 'total_pages': 33, 'page': 0, 'page_label': '209164', 'source_file': 'ABHI.pdf', 'file_type': 'pdf'}, page_content='Received 22 August 2025, accepted 13 October 2025, date of publication 26 November 2025, date of current version 15 December 2025.\nDigital Object Identifier 10.1 109/ACCESS.2025.3637567\nRescue Robots for Casualty Extraction:\nA Comprehensive R

In [44]:
##Test Splitting get into chunks

def split_doc(docs, chunk_size=1000, chunk_overlap=200):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_d = text_splitter.split_documents(docs)
    print(f"Split{len(docs)} documents into {len(split_d)} chunks")

    if split_d:
        print(f"\nExample chunk:")
        print(f"Content: {split_d[0].page_content[:200]} ...")
        print(f"Metadata: {split_d[0].metadata}")
    
    return split_d

In [45]:
chunks = split_doc(all_pdf_doc)

Split112 documents into 368 chunks

Example chunk:
Content: Received 22 August 2025, accepted 13 October 2025, date of publication 26 November 2025, date of current version 15 December 2025.
Digital Object Identifier 10.1 109/ACCESS.2025.3637567
Rescue Robots  ...
Metadata: {'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-11T14:47:25+05:30', 'moddate': '2025-12-12T16:24:36-05:00', 'ieee article id': '11269767', 'trapped': 'False', 'ieee issue id': '10820123', 'subject': 'IEEE Access;2025;13; ;10.1109/ACCESS.2025.3637567', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publication id': '6287639', 'title': 'Rescue Robots for Casualty Extraction: A Comprehensive Review', 'source': '../data/pdf/ABHI.pdf', 'total_pages': 33, 'page': 0, 'page_label': '209164', 'source_file': 'ABHI.pdf', 'file_type': 'pdf'}


Embedding and VectorStoreDB

In [46]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
class EmbeddingM:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        """Initalize the embedding manger
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try: 
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfull. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading model { self.model_name}: {e}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            text: List of text string to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embedding with shape: {embeddings.shape}")
        return embeddings

 

# initialize the embedding manger
embedding_manager = EmbeddingM()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8660.36it/s]


Model loaded successfull. Embedding dimension: 384


/tmp/ipykernel_3840/853427162.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfull. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorDB

In [48]:
class VectorStore:
    """Manges documents embedding in a ChromaDB vector store"""

    def __init__(self, c_name: str = "pdf_documents", persist_directory: str="../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_dicrectory: Directory to presist the vector store
        """

        self.c_name = c_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.c_name,
                metadata={"description": "PDF document embedding for RAG"}
            )

            print(f"Vector store initialized collection: {self.c_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """ 
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embedding for thr documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embedding")
        
        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_indexing']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            
            #Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")


     
        
vectorstore = VectorStore()
vectorstore





Vector store initialized collection: pdf_documents
Existing documents in collection: 1840


In [49]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-11T14:47:25+05:30', 'moddate': '2025-12-12T16:24:36-05:00', 'ieee article id': '11269767', 'trapped': 'False', 'ieee issue id': '10820123', 'subject': 'IEEE Access;2025;13; ;10.1109/ACCESS.2025.3637567', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'ieee publication id': '6287639', 'title': 'Rescue Robots for Casualty Extraction: A Comprehensive Review', 'source': '../data/pdf/ABHI.pdf', 'total_pages': 33, 'page': 0, 'page_label': '209164', 'source_file': 'ABHI.pdf', 'file_type': 'pdf'}, page_content='Received 22 August 2025, accepted 13 October 2025, date of publication 26 November 2025, date of current version 15 December 2025.\nDigital Object Identifier 10.1 109/ACCESS.2025.3637567\nRescue Robots for Casualty Extraction:\nA Comprehensive R

In [50]:
# convert the text to embeddings
text = [doc.page_content for doc in chunks]

# Generate Embedding
embeddings=embedding_manager.generate_embedding(text)

# store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embedding for 368 texts..


Batches: 100%|██████████| 12/12 [00:08<00:00,  1.50it/s]


Generated Embedding with shape: (368, 384)
Adding 368 documents to vector store...
Successfully added 368 documents to vector store
Total documents in collection: 2208


Retriever Pipeline From VectorStore

In [51]:
class RAG:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingM):
        """ 
        Initialize the retriever
        
        Args:
            vector_store: vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(
        self, 
        query: str, 
        top_k: int = 5, 
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        # 🔹 Generate query embedding safely
        embeddings = self.embedding_manager.generate_embedding([query])

        if embeddings is None or len(embeddings) == 0:
            print("❌ Failed to generate embedding")
            return []

        query_embedding = embeddings[0]

        try:
            # 🔹 Query vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            # 🔹 Check if results exist
            if results.get('documents'):
                documents = results.get('documents', [[]])[0]
                metadatas = results.get('metadatas', [[]])[0]
                distances = results.get('distances', [[]])[0]
                ids = results.get('ids', [[]])[0]

                # 🔹 Handle mismatch safely
                min_len = min(len(ids), len(documents), len(metadatas), len(distances))

                for i in range(min_len):
                    doc_id = ids[i]
                    document = documents[i]
                    metadata = metadatas[i]
                    distance = distances[i]

                    # 🔹 Convert distance → similarity score
                    similarity_score = 1 / (1 + distance)

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"✅ Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("⚠️ No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"❌ Error during retrieval: {e}")
            return []

 
ragretriever = RAG(vectorstore, embedding_manager)

 

In [52]:
ragretriever.retrieve ("What is multi modal driver anger emotion recognization")

Retrieving documents for query: 'What is multi modal driver anger emotion recognization'
Top k: 5, Score threshold: 0.0
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.58it/s]

Generated Embedding with shape: (1, 384)
✅ Retrieved 5 documents (after filtering)


[{'id': 'doc_34a56974_241',
  'content': 'A  multimodal  fusion  layer  combines  these  independent  features,  enabling  the  model  to  capture  complex  emotional  variations  that  single-modal  systems  fail  to  identify.  Their  results  showed  that  integrating  multiple  data  streams  significantly  enhances  recognition  accuracy,  particularly  in  identifying  high-intensity  emotions  such  as  anger.  This  work  forms  a  foundational  basis  for  understanding  the  importance  of  multi-sensor  fusion  in  driver  anger  detection.  \n2.2.  Driver  Emotion  Recognition  Using  Facial  Features  (Li  &  Chen,  2020)',
  'metadata': {'doc_indexing': 241,
   'content_length': 599,
   'source': '../data/pdf/Major Document.pdf',
   'page_label': '11',
   'total_pages': 78,
   'creationdate': '',
   'source_file': 'Major Document.pdf',
   'producer': 'Skia/PDF m148 Google Docs Renderer',
   'page': 10,
   'creator': 'PyPDF',
   'title': 'Major Document',
   'file_type': '

Integration VrctorDB Context pipeline with LLM output

In [53]:
#simple rag pipeline with groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
load_dotenv()

#Initialize the Groq LLM
key = os.getenv("key")

llm = ChatGroq(groq_api_key=key, model_name = "llama-3.3-70b-versatile", temperature=0.1,max_tokens=1024)

# Retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    #retriever the context
    results = retriever.retrieve(query,top_k)
    content ="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not content :
        return "No revelent context to answer the question concisely"
    
    #generate the answer
    prompt= """Use the following contextcontex to anser the question concisely.
    Content: {content}
    Question: {query}
    Answer: """

    response =llm.invoke([HumanMessage(content=prompt.format(content=content, query=query))])
    return response.content

In [54]:
answer = rag_simple("What is Machine learning?",ragretriever,llm)
print(answer)

Retrieving documents for query: 'What is Machine learning?'
Top k: 3, Score threshold: 0.0
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 167.20it/s]

Generated Embedding with shape: (1, 384)
✅ Retrieved 3 documents (after filtering)


Machine learning is not explicitly defined in the context, but based on the information provided, it can be inferred that machine learning refers to a set of algorithms used to enable intelligent robot behavior, particularly through reinforcement learning.


Enhanced RAG Pipeline features

In [57]:
# Enhanced RAG pipeline features
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    -   Return answer, sources, confidence score, and full context"""

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources':[], 'context':''}
    
    #prepare context and sourcess
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
        'page' : doc['metadata'].get('page','unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300]+'...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence':confidence
    }
    if return_context:
        output['context']=context
    return output

In [58]:
result= rag_advanced('what is Machine Learning', ragretriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is Machine Learning'
Top k: 3, Score threshold: 0.1
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 95.11it/s]

Generated Embedding with shape: (1, 384)
✅ Retrieved 3 documents (after filtering)


Answer: Machine learning refers to the use of algorithms that enable machines to learn from data and improve their performance on a task, such as robotics, without being explicitly programmed.
Sources: [{'source': 'ABHI.pdf', 'page': 32, 'score': 0.46907780608058147, 'preview': 'computational intelligence from Tokyo Institute of\nTechnology. He was a Visiting Senior Research\nFellow with the King’s College London, and\na Research Team Leader with the Advanced\nRobotics Department, Italian Institute of Technol-\nogy (IIT). He is currently an Associate Professor\nof robotics with t...'}, {'source': 'ABHI.pdf', 'page': 32, 'score': 0.46907780608058147, 'preview': 'computational intelligence from Tokyo Institute of\nTechnology. He was a Visiting Senior Research\nFellow with the King’s College London, and\na Research Team Leader with the Advanced\nRobotics Department, Italian Institute of Technol-\nogy (IIT). He is currently an Associate Professor\nof robotics with t...'}, {'source': 'ABHI.

In [69]:
# Advanced RAG Pipeline: Streaming, Citations, History, Summariztion...
from typing import List, Dict, Any
import time

class AdvancedRagPipeline:
    def __init__(self,retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = [] #store query history


    def query(self, question:str, top_k: int = 5, min_score: float = 0.2, stream : bool=False)-> Dict[str, Any]:
            start_time = time.time()

            # Retrieve documents
            results = self.retriever.retrieve(question, top_k)

            #Filter by score
            filtered_docs = []
            for doc in results:
                score = doc.get("score", 1.0) if isinstance(doc, dict) else 1.0
                if score >= min_score:
                    filtered_docs.append(doc)

            # extract context safely
            context_parts = []
            for doc in filtered_docs:
                if isinstance(doc,dict):
                    context_parts.append(doc.get("content") or doc.get("text") or "")
                else:
                    context_parts.append(getattr(doc,"page_content", ""))

            context = "\n\n".join(context_parts).strip()
            if not context:
                return {
                    "answer":"No relevant context found",
                    "sources":[],
                    "time":0
                }
            
            #prompt
            prompt = f"""Your are a helpful AI assistant.
            Use only the given context to answer. IF the answer is not in the context, say "I don't know".
            Context:{context}
            Question:{question}
            Answer:"""    

            # LLM call
            if stream:
                response = self.llm.stram(prompt)
                answer=""
                for chunk in response:
                    print(chunk.content, end="", flush= True)
                    answer += chunk.content
            else:
                response = self.llm.invoke(prompt)
                answer = response.content

            #extact Sources
            sources=[]
            for doc in filtered_docs:
                if isinstance(doc, dict):
                    sources.append(doc.get("source", "unknown"))
                else:
                    sources.append(getattr(doc,"metadata", {}).get("source", "unknown"))     

            #Save History
            self.history.append({
                "question":question,
                "answer":answer
            })

            end_time = time.time()

            return {
                "answer" : answer,
                "sources" : sources,
                "time" : round(end_time - start_time, 2)
            }

In [70]:
answer = AdvancedRagPipeline(ragretriever, llm)
result = answer.query("What is multi-modal driver system")
print(result)

Retrieving documents for query: 'What is multi-modal driver system'
Top k: 5, Score threshold: 0.0
Generating embedding for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.06it/s]

Generated Embedding with shape: (1, 384)
✅ Retrieved 5 documents (after filtering)


{'answer': "The Context-Aware Multimodal Driver Anger Recognition System (CA-MDER) is a system that captures a comprehensive representation of the driver's emotional state through a multimodal approach, collecting data from different sources such as facial images or video, speech signals, and vehicle driving data.", 'sources': ['unknown', 'unknown', 'unknown', 'unknown', 'unknown'], 'time': 0.83}
